# transdiagnostic subtyping

In [ ]:
# Transdiagnostic Subtyping of Neurodivergent Populations
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import HDBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.semi_supervised import LabelPropagation
import umap
import warnings
warnings.filterwarnings('ignore')

print("=== Transdiagnostic Subtyping Experiment ===\n")

# Load datasets
c4_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/data_c4_matched_balanced.csv')
ybt_balanced = pd.read_csv('/Users/eb2007/playground/bullpy/c4_play2/data/processed/YBT_balanced_standardized.csv')

print(f"C4 Dataset: {c4_balanced.shape}")
print(f"YBT Dataset: {ybt_balanced.shape}")

# 2. feature engineering for transdiagnostic analysis 

In [ ]:
# Apply existing feature engineering from dl_domain_adaption.ipynb
def create_aggregate_features(df, prefix, n_items):
    item_cols = [f"{prefix}_{i}" for i in range(1, n_items+1) if f"{prefix}_{i}" in df.columns]
    if item_cols:
        df[f"{prefix}_total"] = df[item_cols].sum(axis=1)
    return df

for prefix, n_items in [('eq', 10), ('aq', 10), ('sqr', 10), ('spq', 10)]:
    c4_balanced = create_aggregate_features(c4_balanced, prefix, n_items)
    ybt_balanced = create_aggregate_features(ybt_balanced, prefix, n_items)

# D-score (Empathy - Social Responsiveness)
for df in [c4_balanced, ybt_balanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['d_score'] = df['eq_total'] - df['sqr_total']

# Age interactions
for df in [c4_balanced, ybt_balanced]:
    if 'age' in df.columns and 'aq_total' in df.columns:
        df['age_x_aq'] = df['age'] * df['aq_total']
    if 'age' in df.columns and 'eq_total' in df.columns:
        df['age_x_eq'] = df['age'] * df['eq_total']

# Trait interactions
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_interaction'] = df['aq_total'] * df['eq_total']

# Cognitive profile ratios
for df in [c4_balanced, ybt_balanced]:
    if 'eq_total' in df.columns and 'sqr_total' in df.columns:
        df['eq_sqr_ratio'] = df['eq_total'] / (df['sqr_total'] + 1e-8)
    if 'aq_total' in df.columns and 'eq_total' in df.columns:
        df['aq_eq_ratio'] = df['aq_total'] / (df['eq_total'] + 1e-8)

# Log transformations
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns:
        df['log_aq_total'] = np.log1p(np.clip(df['aq_total'], a_min=0, a_max=None))
    if 'age' in df.columns:
        df['sqrt_age'] = np.sqrt(np.clip(df['age'], a_min=0, a_max=None))

# High trait flags
for df in [c4_balanced, ybt_balanced]:
    if 'aq_total' in df.columns:
        df['high_aq'] = (df['aq_total'] > 32).astype(int)
    if 'eq_total' in df.columns:
        df['low_eq'] = (df['eq_total'] < 30).astype(int)

# 3. feature selection and data prep

In [ ]:
# Define transdiagnostic features
transdiagnostic_features = [
    'aq_total', 'eq_total', 'sqr_total', 'spq_total',
    'd_score', 'aq_eq_interaction', 'eq_sqr_ratio', 'aq_eq_ratio',
    'log_aq_total', 'sqrt_age', 'high_aq', 'low_eq'
]

demographic_features = ['age', 'sex_num'] if 'sex_num' in c4_balanced.columns else ['age']
selected_features = transdiagnostic_features + demographic_features

# Filter available features
c4_features = [f for f in selected_features if f in c4_balanced.columns]
ybt_features = [f for f in selected_features if f in ybt_balanced.columns]

print(f"Selected features for C4: {len(c4_features)}")
print(f"Selected features for YBT: {len(ybt_features)}")

# Find common features between datasets
common_features = list(set(c4_features) & set(ybt_features))
print(f"Common features: {len(common_features)}")
print(f"Missing in YBT: {set(c4_features) - set(ybt_features)}")
print(f"Missing in C4: {set(ybt_features) - set(c4_features)}")

# Use only common features for consistent scaling
c4_data = c4_balanced[common_features].copy()
ybt_data = ybt_balanced[common_features].copy()
c4_targets = c4_balanced['autism_target'] if 'autism_target' in c4_balanced.columns else None
ybt_targets = ybt_balanced['autism_target'] if 'autism_target' in ybt_balanced.columns else None

# Handle missing values and standardize
c4_data = c4_data.fillna(c4_data.mean())
ybt_data = ybt_data.fillna(ybt_data.mean())

scaler = StandardScaler()
c4_scaled = scaler.fit_transform(c4_data)
ybt_scaled = scaler.transform(ybt_data)

print(f"C4 scaled data shape: {c4_scaled.shape}")
print(f"YBT scaled data shape: {ybt_scaled.shape}")

# 4. dimensionality reduction and clustering

In [ ]:
# PCA for visualization
pca = PCA(n_components=2)
c4_pca = pca.fit_transform(c4_scaled)
ybt_pca = pca.transform(ybt_scaled)

# UMAP for clustering
umap_reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
c4_umap = umap_reducer.fit_transform(c4_scaled)
ybt_umap = umap_reducer.transform(ybt_scaled)

# HDBSCAN clustering
hdbscan = HDBSCAN(min_cluster_size=50, min_samples=5)
c4_hdbscan_labels = hdbscan.fit_predict(c4_scaled)
ybt_hdbscan_labels = hdbscan.fit_predict(ybt_scaled)

print(f"C4 HDBSCAN clusters: {len(set(c4_hdbscan_labels)) - 1}")
print(f"YBT HDBSCAN clusters: {len(set(ybt_hdbscan_labels)) - 1}")

# Gaussian Mixture Models
gmm = GaussianMixture(n_components=4, random_state=42)
c4_gmm_labels = gmm.fit_predict(c4_scaled)
ybt_gmm_labels = gmm.fit_predict(ybt_scaled)

# 5. semi-supervised learning

In [ ]:
# Semi-supervised learning with diagnosis labels
if ybt_targets is not None:
    label_prop = LabelPropagation(kernel='knn', n_neighbors=10)
    
    # Create semi-supervised labels
    semi_labels = ybt_targets.copy()
    mask = np.random.choice([True, False], size=len(semi_labels), p=[0.3, 0.7])
    semi_labels[mask] = -1  # Unlabeled
    
    label_prop.fit(ybt_scaled, semi_labels)
    ybt_semi_labels = label_prop.predict(ybt_scaled)
    
    print(f"Semi-supervised clusters: {len(set(ybt_semi_labels))}")

# 6. visualization and analysis 

In [ ]:
# Plot clustering results
def plot_clusters(data_2d, labels, title, dataset_name):
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 2, 1)
    plt.scatter(data_2d[:, 0], data_2d[:, 1], c=labels, cmap='tab10', alpha=0.6)
    plt.title(f'{title} - {dataset_name}')
    plt.xlabel('Component 1')
    plt.ylabel('Component 2')
    
    plt.subplot(2, 2, 2)
    unique_labels, counts = np.unique(labels, return_counts=True)
    plt.bar(unique_labels, counts)
    plt.title('Cluster Distribution')
    plt.xlabel('Cluster')
    plt.ylabel('Count')
    
    plt.tight_layout()
    plt.show()

# Plot results
plot_clusters(c4_pca, c4_hdbscan_labels, "HDBSCAN Clustering", "C4 Dataset")
plot_clusters(ybt_pca, ybt_hdbscan_labels, "HDBSCAN Clustering", "YBT Dataset")

# 7. cluster characterization 

In [ ]:
def characterize_clusters(data, labels, feature_names, dataset_name):
    print(f"\nCluster Characterization for {dataset_name}:")
    
    for cluster_id in sorted(set(labels)):
        if cluster_id == -1:
            continue
        
        cluster_mask = labels == cluster_id
        cluster_data = data[cluster_mask]
        
        print(f"\nCluster {cluster_id} (n={len(cluster_data)}):")
        
        cluster_means = np.mean(cluster_data, axis=0)
        feature_importance = np.abs(cluster_means - np.mean(data, axis=0))
        top_features_idx = np.argsort(feature_importance)[-5:]
        
        for idx in top_features_idx:
            feature_name = feature_names[idx] if idx < len(feature_names) else f"Feature_{idx}"
            cluster_mean = cluster_means[idx]
            overall_mean = np.mean(data[:, idx])
            print(f"  {feature_name}: {cluster_mean:.3f} (vs {overall_mean:.3f} overall)")

# Characterize clusters
characterize_clusters(c4_scaled, c4_hdbscan_labels, c4_features, "C4 Dataset")
characterize_clusters(ybt_scaled, ybt_hdbscan_labels, ybt_features, "YBT Dataset")

# poor results - debugging

In [ ]:
# Check if your "balanced" datasets are actually balanced
print("=== DATA BALANCE CHECK ===")
print(f"C4 autism_target distribution:")
print(c4_balanced['autism_target'].value_counts())
print(f"\nYBT autism_target distribution:")
print(ybt_balanced['autism_target'].value_counts())

# Check raw questionnaire scores
print(f"\n=== RAW SCORE DISTRIBUTIONS ===")
print("C4 AQ scores:")
print(c4_balanced['aq_total'].describe())
print(f"\nC4 autism rate: {c4_balanced['autism_target'].mean():.3f}")

print(f"\nYBT AQ scores:")
print(ybt_balanced['aq_total'].describe())
print(f"\nYBT autism rate: {ybt_balanced['autism_target'].mean():.3f}")

In [ ]:
# Plot raw AQ vs EQ scores to see natural clusters
plt.figure(figsize=(15, 5))

plt.subplot(1, 3, 1)
plt.scatter(c4_balanced['aq_total'], c4_balanced['eq_total'], 
           c=c4_balanced['autism_target'], alpha=0.6)
plt.xlabel('AQ Total')
plt.ylabel('EQ Total')
plt.title('C4: AQ vs EQ (colored by autism diagnosis)')
plt.colorbar()

plt.subplot(1, 3, 2)
plt.scatter(ybt_balanced['aq_total'], ybt_balanced['eq_total'], 
           c=ybt_balanced['autism_target'], alpha=0.6)
plt.xlabel('AQ Total')
plt.ylabel('EQ Total')
plt.title('YBT: AQ vs EQ (colored by autism diagnosis)')
plt.colorbar()

plt.subplot(1, 3, 3)
plt.scatter(c4_balanced['aq_total'], c4_balanced['sqr_total'], 
           c=c4_balanced['autism_target'], alpha=0.6)
plt.xlabel('AQ Total')
plt.ylabel('SQR Total')
plt.title('C4: AQ vs SQR (colored by autism diagnosis)')
plt.colorbar()

plt.tight_layout()
plt.show()

# simpler clustering 

In [ ]:
# Try k-means with k=2 (autism vs non-autism)
from sklearn.cluster import KMeans

print("=== K-MEANS CLUSTERING ===")
kmeans = KMeans(n_clusters=2, random_state=42)
c4_kmeans = kmeans.fit_predict(c4_scaled)
ybt_kmeans = kmeans.fit_predict(ybt_scaled)

# Check if k-means clusters align with autism diagnosis
print("C4 K-means vs Autism diagnosis:")
for i in range(2):
    cluster_mask = c4_kmeans == i
    autism_rate = np.mean(c4_targets[cluster_mask])
    print(f"Cluster {i}: Autism rate = {autism_rate:.3f}")

print("\nYBT K-means vs Autism diagnosis:")
for i in range(2):
    cluster_mask = ybt_kmeans == i
    autism_rate = np.mean(ybt_targets[cluster_mask])
    print(f"Cluster {i}: Autism rate = {autism_rate:.3f}")

# checkling feature correlations

In [ ]:
# Check if your features are too correlated
print("=== FEATURE CORRELATIONS ===")
corr_matrix = c4_data.corr().abs()
print("C4 feature correlations:")
print(corr_matrix)

# Check feature variance
print(f"\nFeature variances:")
print(c4_data.var().sort_values(ascending=False))

# trying different HDBSCAN

In [ ]:
# Try more aggressive clustering
print("=== HDBSCAN PARAMETER TESTING ===")

# Test different min_cluster_size values
for min_size in [10, 20, 50, 100]:
    hdbscan_test = HDBSCAN(min_cluster_size=min_size, min_samples=5)
    labels = hdbscan_test.fit_predict(c4_scaled)
    n_clusters = len(set(labels)) - 1  # -1 for noise
    print(f"min_cluster_size={min_size}: {n_clusters} clusters")

# check if standardization is the problem 

In [ ]:
# Try clustering without standardization
print("=== CLUSTERING WITHOUT STANDARDIZATION ===")
c4_unscaled = c4_data.values
hdbscan_unscaled = HDBSCAN(min_cluster_size=50, min_samples=5)
labels_unscaled = hdbscan_unscaled.fit_predict(c4_unscaled)
print(f"Unscaled clusters: {len(set(labels_unscaled)) - 1}")

# focus on core features only 

In [ ]:
# Try with just the most important features
core_features = ['aq_total', 'eq_total', 'sqr_total']
c4_core = c4_balanced[core_features].copy()
ybt_core = ybt_balanced[core_features].copy()

# Standardize and cluster
scaler_core = StandardScaler()
c4_core_scaled = scaler_core.fit_transform(c4_core)
ybt_core_scaled = scaler_core.transform(ybt_core)

hdbscan_core = HDBSCAN(min_cluster_size=50, min_samples=5)
c4_core_labels = hdbscan_core.fit_predict(c4_core_scaled)
print(f"Core features clusters: {len(set(c4_core_labels)) - 1}")